# Tutorial: The Data Science Landscape

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work.  
> **Time:** ~1.5 hours  
> **Prerequisites:** Python basics, pip  
> **What you'll build:** An interactive timeline of AI milestones, plus your first data analysis: 30 years of machine learning paper publication trends on arXiv.

---

## What We're Building

We're going to do two things:

1. Build a visual timeline of the AI milestones from `explanation.md`. A picture of history is easier to hold in your head than a list.
2. Analyse a dataset of arXiv machine learning paper counts. You'll see, with numbers, when different subfields took off, and you'll use basic pandas operations to find the story in the data.

This is also your first taste of the core data science workflow: **load → look → clean → visualise → interpret**. We'll do all five steps here, and we'll repeat them in every module that follows.

## Step 0: Setup

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install pandas matplotlib seaborn requests

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
from datetime import datetime

# Make plots look good
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Setup complete.')

## Part 1: The AI Timeline

Let's build the timeline from `explanation.md` as a proper visualisation. We'll define the milestones ourselves. That gives you practice creating structured data in Python before you load any external datasets.

### Step 1: Define the Milestones

In [ ]:
# Each milestone has a year, a label, and an era
# Era determines the colour on the chart
milestones = [
    # Era 1: Rules & Expert Systems
    {"year": 1950, "label": "Turing Test proposed", "era": "Rules & Expert Systems"},
    {"year": 1956, "label": "'Artificial Intelligence' coined\n(Dartmouth Conference)", "era": "Rules & Expert Systems"},
    {"year": 1965, "label": "DENDRAL: first expert system", "era": "Rules & Expert Systems"},
    {"year": 1972, "label": "MYCIN: medical diagnosis AI", "era": "Rules & Expert Systems"},
    
    # Era 2: Statistical ML
    {"year": 1986, "label": "Backpropagation\nrediscovered", "era": "Statistical ML"},
    {"year": 1995, "label": "Support Vector Machines", "era": "Statistical ML"},
    {"year": 1997, "label": "LSTM invented", "era": "Statistical ML"},
    {"year": 2001, "label": "Random Forests", "era": "Statistical ML"},
    {"year": 2001, "label": "AdaBoost → Gradient Boosting", "era": "Statistical ML"},
    {"year": 2009, "label": "ImageNet dataset released", "era": "Statistical ML"},
    
    # Era 3: Deep Learning & Foundation Models
    {"year": 2012, "label": "AlexNet wins ImageNet", "era": "Deep Learning & Foundation Models"},
    {"year": 2014, "label": "GANs", "era": "Deep Learning & Foundation Models"},
    {"year": 2017, "label": "Transformer architecture\n('Attention Is All You Need')", "era": "Deep Learning & Foundation Models"},
    {"year": 2018, "label": "BERT (Google)", "era": "Deep Learning & Foundation Models"},
    {"year": 2020, "label": "GPT-3", "era": "Deep Learning & Foundation Models"},
    {"year": 2021, "label": "GitHub Copilot", "era": "Deep Learning & Foundation Models"},
    {"year": 2022, "label": "ChatGPT", "era": "Deep Learning & Foundation Models"},
    {"year": 2023, "label": "GPT-4, Llama, Claude", "era": "Deep Learning & Foundation Models"},
]

df_milestones = pd.DataFrame(milestones)
print(f"{len(df_milestones)} milestones across {df_milestones['era'].nunique()} eras")
df_milestones.head()

### Step 2: Plot the Timeline

In [ ]:
era_colours = {
    "Rules & Expert Systems":         "#4E79A7",
    "Statistical ML":                  "#F28E2B",
    "Deep Learning & Foundation Models": "#E15759",
}

fig, ax = plt.subplots(figsize=(16, 5))

# Draw the baseline
ax.axhline(0, color='black', linewidth=1.5, zorder=1)

# Alternate label positions above/below the line to avoid overlaps
positions = [0.6, 1.1, 1.6, 2.1, -0.6, -1.1, -1.6]

for i, row in df_milestones.iterrows():
    colour = era_colours[row['era']]
    y_pos = positions[i % len(positions)]
    
    # Dot on the line
    ax.scatter(row['year'], 0, color=colour, s=80, zorder=3)
    
    # Vertical connector
    ax.plot([row['year'], row['year']], [0, y_pos * 0.85], 
            color=colour, linewidth=1, linestyle='--', alpha=0.6, zorder=2)
    
    # Label
    ax.text(row['year'], y_pos, row['label'],
            ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                      edgecolor=colour, linewidth=1.2))

# Era background shading
ax.axvspan(1950, 1984, alpha=0.04, color=era_colours["Rules & Expert Systems"])
ax.axvspan(1984, 2011, alpha=0.04, color=era_colours["Statistical ML"])
ax.axvspan(2011, 2025, alpha=0.04, color=era_colours["Deep Learning & Foundation Models"])

# Legend
legend_patches = [mpatches.Patch(color=c, label=e) for e, c in era_colours.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

ax.set_xlim(1945, 2026)
ax.set_ylim(-2.5, 2.5)
ax.set_yticks([])
ax.set_xlabel('Year', fontsize=12)
ax.set_title('Key Milestones in AI / Data Science', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('ai_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to ai_timeline.png')

> **What just happened?** We built a structured dataset from scratch, assigned categories (eras), and visualised it. You'll use the same pattern for building features and plotting distributions throughout the course.

---

## Part 2: Exploring arXiv ML Paper Trends

Now we'll look at paper counts. We'll use a dataset of arXiv machine learning paper counts by year and sub-category. It shows when different parts of the field took off.

This dataset is representative, not downloaded: we construct it here from known arXiv trends, so the shape is right but individual counts are approximate. In a later module you'll learn to call APIs and pull real sources.

### Step 3: Load the Data

In [ ]:
# Approximate paper counts per year per ML subfield (thousands of papers)
# Based on known arXiv growth patterns
data = {
    'year': list(range(1993, 2025)),
    'classical_ml': [
        0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.5,
        2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.8, 6.5, 7.0,
        7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0,
        12.5, 13.0
    ],
    'deep_learning': [
        0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.1, 0.2, 0.2, 0.3,
        0.4, 0.5, 0.6, 0.8, 1.0, 2.0, 5.0, 12.0, 22.0, 35.0,
        50.0, 70.0, 95.0, 130.0, 170.0, 210.0, 260.0, 320.0, 390.0, 460.0,
        550.0, 650.0
    ],
    'nlp_llm': [
        0.0, 0.0, 0.0, 0.1, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6,
        0.8, 1.0, 1.5, 2.0, 3.0, 4.5, 7.0, 12.0, 20.0, 30.0,
        45.0, 65.0, 100.0, 160.0, 230.0, 310.0, 410.0, 520.0, 650.0, 800.0,
        980.0, 1200.0
    ],
    'computer_vision': [
        0.0, 0.1, 0.1, 0.2, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8,
        1.0, 1.2, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0, 35.0, 55.0,
        80.0, 110.0, 150.0, 200.0, 260.0, 330.0, 410.0, 500.0, 600.0, 720.0,
        850.0, 1000.0
    ],
    'reinforcement_learning': [
        0.0, 0.0, 0.1, 0.1, 0.2, 0.2, 0.3, 0.4, 0.5, 0.6,
        0.7, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0, 9.0, 16.0, 25.0,
        38.0, 55.0, 75.0, 100.0, 130.0, 165.0, 205.0, 250.0, 300.0, 355.0,
        415.0, 480.0
    ],
}

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"Years covered: {df['year'].min()} – {df['year'].max()}")
df.head(10)

### Step 4: Understand the Data

Before plotting anything, always look at the shape, the types, and basic statistics. This habit will save you hours of debugging.

In [ ]:
# Basic inspection
print("=== Data types ===")
print(df.dtypes)

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Summary statistics ===")
df.describe().round(1)

In [ ]:
# Which subfield had the most papers in the most recent year?
latest_year = df[df['year'] == df['year'].max()]
subfields = ['classical_ml', 'deep_learning', 'nlp_llm', 'computer_vision', 'reinforcement_learning']

print(f"Paper counts (thousands) in {df['year'].max()}:")
for col in subfields:
    print(f"  {col:35s}: {latest_year[col].values[0]:>8.0f}k")

### Step 5: Visualise the Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colours = ['#4E79A7', '#E15759', '#F28E2B', '#76B7B2', '#59A14F']
labels  = ['Classical ML', 'Deep Learning', 'NLP / LLMs', 'Computer Vision', 'Reinforcement Learning']

# --- Left: Full history ---
for col, colour, label in zip(subfields, colours, labels):
    axes[0].plot(df['year'], df[col], label=label, color=colour, linewidth=2)

# Mark the transformer paper
axes[0].axvline(2017, color='grey', linestyle=':', linewidth=1.5)
axes[0].text(2017.2, axes[0].get_ylim()[1] * 0.85, 'Transformer\n(2017)', 
             fontsize=9, color='grey')

axes[0].set_title('ML Paper Volume by Subfield\n(all years)', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Papers published (thousands, approx.)')
axes[0].legend(fontsize=9)

# --- Right: Zoom into 2012–2024 ---
df_modern = df[df['year'] >= 2012].copy()

for col, colour, label in zip(subfields, colours, labels):
    axes[1].plot(df_modern['year'], df_modern[col], label=label, color=colour, 
                 linewidth=2, marker='o', markersize=4)

axes[1].axvline(2017, color='grey', linestyle=':', linewidth=1.5)
axes[1].axvline(2022, color='darkred', linestyle=':', linewidth=1.5)
axes[1].text(2017.1, axes[1].get_ylim()[1] * 0.95, 'Transformer', fontsize=8, color='grey')
axes[1].text(2022.1, axes[1].get_ylim()[1] * 0.85, 'ChatGPT', fontsize=8, color='darkred')

axes[1].set_title('ML Paper Volume by Subfield\n(2012–2024 zoom)', fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('arxiv_trends.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** You just did your first data analysis: load → inspect → visualise → interpret. The chart shows two things clearly. First, deep learning, NLP, and computer vision grow exponentially after 2012, and again after 2017. Second, classical ML keeps growing, but slowly. It isn't being replaced. It's being dwarfed.

### Step 6: Answer a Specific Question From the Data

Data analysis isn't only plotting. It's answering questions. Let's answer one: **in what year did deep learning paper volume first exceed classical ML?**

In [ ]:
# Find the crossover year
df['dl_vs_classical'] = df['deep_learning'] - df['classical_ml']
crossover = df[df['dl_vs_classical'] > 0].iloc[0]

print(f"Deep learning overtook classical ML in: {int(crossover['year'])}")
print(f"  Classical ML papers that year: {crossover['classical_ml']:.1f}k")
print(f"  Deep learning papers that year: {crossover['deep_learning']:.1f}k")

# What was the ratio by 2024?
row_2024 = df[df['year'] == 2024].iloc[0]
ratio = row_2024['deep_learning'] / row_2024['classical_ml']
print(f"\nBy 2024, deep learning papers outnumber classical ML by {ratio:.0f}x")

### Step 7: Year-on-Year Growth Rate

In [ ]:
# Calculate year-on-year percentage growth for each subfield
growth = df[['year'] + subfields].copy()
for col in subfields:
    growth[f'{col}_yoy'] = growth[col].pct_change() * 100

# Show peak growth years for NLP/LLMs
nlp_growth = growth[['year', 'nlp_llm_yoy']].dropna()
peak_growth_years = nlp_growth.nlargest(5, 'nlp_llm_yoy')

print("Top 5 years for NLP/LLM paper growth:")
print(peak_growth_years.to_string(index=False))

In [ ]:
# Exclude the inf artifact before ranking growth years
finite_growth = nlp_growth[nlp_growth['nlp_llm_yoy'] != float('inf')]
peak_finite = finite_growth.nlargest(5, 'nlp_llm_yoy')
print("Top 5 years for NLP/LLM paper growth, inf excluded:")
print(peak_finite.to_string(index=False))

> **What just happened?** With the `inf` artefact excluded, 1998 is the largest growth year (100%), but that's an early low-base effect on tiny paper counts. Next comes a cluster at 2009-2011 (56-71%) and another at 2016 (60%). That later cluster sits just before the two inflection points (ImageNet, the Transformer paper) this chapter's timeline already marks.

---

## What You Built

You built two things: a visual history of AI from structured data you defined yourself, and a trend analysis of ML research growth. More importantly, you ran through the core data science loop, **load → inspect → clean → visualise → interpret**, for the first time. Every later module repeats this loop on a different domain.

## What's Next

- Read `explanation.md` if you haven't. The historical context makes the trend charts much more meaningful.
- Open `howto.md` for pandas and matplotlib recipes you'll reuse constantly.
- Keep `reference.md` open as a pandas cheat sheet.
- **Extension challenge:** add a 6th subfield to the milestone list: generative AI and diffusion models. Add it to the DataFrame and update both charts.

---

**Next module:** `02_business_problem_framing`, where you learn to turn a customer's request into a data science problem statement before writing a single line of code.